In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")
import mlflow 
from imblearn.combine import SMOTETomek

In [2]:
#Step1 create an imbalanced binary classification dataset

X,y=make_classification(n_samples=1000,
                       n_features=10,
                       n_informative=2,
                       n_redundant=8,
                       weights=[0.9,0.1],
                       flip_y=0,
                       random_state=42)

np.unique(y,return_counts=True)

(array([0, 1]), array([900, 100], dtype=int64))

In [3]:
X_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

## Experiment 1: Train Logistic Regression Classifier

In [4]:
log_reg=LogisticRegression(C=1,solver="liblinear")
log_reg.fit(X_train,y_train)
y_pred_log_reg=log_reg.predict(x_test)
print(classification_report(y_test,y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



## Experiment 2 : Train Random Forest Classifier

In [5]:
rf_clf=RandomForestClassifier(n_estimators=30,max_depth=3)
rf_clf.fit(X_train,y_train)
y_pred_rf_clf=rf_clf.predict(x_test)
print(classification_report(y_test,y_pred_rf_clf))


              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



## Experiment 3: Train XGBoost Classifier

In [6]:
xgb_clf=XGBClassifier(use_label_encoder=False,eval_metric="logloss")
xgb_clf.fit(X_train,y_train)
y_pred_xgb_clf=xgb_clf.predict(x_test)
print(classification_report(y_test,y_pred_xgb_clf))


              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



## Experiment 4: Handle Class Imbalance using SMOTETomek and then Train XGBoost

In [7]:
np.unique(y_train,return_counts=True)

(array([0, 1]), array([630,  70], dtype=int64))

In [8]:
smt=SMOTETomek(random_state=42)
X_train_smt,y_train_smt=smt.fit_resample(X_train,y_train)
np.unique(y_train_smt,return_counts=True)

(array([0, 1]), array([619, 619], dtype=int64))

In [9]:
xgb_clf=XGBClassifier(use_label_encoder=False,eval_metric="logloss")
xgb_clf.fit(X_train_smt,y_train_smt)
y_pred_xgb_clf=xgb_clf.predict(x_test)
print(classification_report(y_test,y_pred_xgb_clf))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



**🔹 What is a Label Encoder?**

A label encoder is used to convert categorical values into numbers.

Example:
"Yes" → 1 

"No"  → 0

or

"Low" → 0 

"Medium" → 1 

"High" → 2

👉 Models cannot understand text, so we convert them into numbers.

🔸 So when you write:

`use_label_encoder=False`

👉 You are telling XGBoost:

“Don’t do any automatic encoding. I will handle it.”

## Track Experiments using MLFlow

In [10]:
models=[
(
"Logistic Regression",
 {"C":1,"solver":"liblinear"},
 LogisticRegression(),
(X_train,y_train),
(x_test,y_test)
),
(
"Random Forest",
{"n_estimators":30,"max_depth":3},
    RandomForestClassifier(),
(X_train,y_train),
(x_test,y_test)
),
(
 "XGBClassifier",
  {"use_label_encoder":False,"eval_metric":"logloss"},
    XGBClassifier(),
  (X_train,y_train),
(x_test,y_test)  
),
(
"XGBClassifier with SMOTE",
{"use_label_encoder":False,"eval_metric":"logloss"},
XGBClassifier(),
(X_train_smt,y_train_smt),
(x_test,y_test)      
)
]

In [11]:
reports=[]

for model_name,params,model, train_set, test_set in models:
    X_train=train_set[0]
    y_train=train_set[1]
    x_test=test_set[0]
    y_test=test_set[1]
    
    model.set_params(**params)
    model.fit(X_train,y_train)
    y_pred=model.predict(x_test)
    report=classification_report(y_test,y_pred,output_dict=True)
    reports.append(report)

In [12]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9607142857142857,
   'recall': 0.9962962962962963,
   'f1-score': 0.9781818181818182,
   'support': 270.0},
  '1': {'precision': 0.95,
   'recall': 0.6333333333333333,
   'f1-score': 0.76,
   'support': 30.0},
  'accuracy': 0.96,
  'macro avg': {'precision': 0.9553571428571428,
   'recall': 0.8148148148148149,
   'f1-score': 0.8690909090909091,
   'support': 300.0},
  'weighted avg': {'precision': 0.9596428571428572,
   'recall':

In [13]:
pip install dagshub

Note: you may need to restart the kernel to use updated packages.


In [14]:
# dagshub setup

import dagshub
dagshub.init(repo_owner='elzbeth1', repo_name='mlflow_dagshub_demo', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=77787fdf-b946-40aa-8de5-7ea483802b2c&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=bcce31b14a290ec389675852d5f2c30eba1fff2b9ff07697fe5daf7a5e145eae




Accessing as elzbeth1

Initialized MLflow to track repo "elzbeth1/mlflow_dagshub_demo"

Repository elzbeth1/mlflow_dagshub_demo initialized!

In [15]:
#mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_tracking_uri("https://dagshub.com/elzbeth1/mlflow_dagshub_demo.mlflow")
mlflow.set_experiment("Anomaly Detection")

2026/03/25 19:57:33 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/87831e6e983e492bafb215ed5efe7d53', creation_time=1774448853612, experiment_id='0', last_update_time=1774448853612, lifecycle_stage='active', name='Anomaly Detection', tags={}, workspace='default'>

In [16]:
print(mlflow.get_tracking_uri())

https://dagshub.com/elzbeth1/mlflow_dagshub_demo.mlflow


In [14]:
for i,element in enumerate(models):
    model_name=element[0]
    params=element[1]
    model=element[2]
    report=reports[i]
    
    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metric('accuracy',report['accuracy'])
        mlflow.log_metric('recall_class_0',report['0']['recall'])
        mlflow.log_metric('recall_class_1',report['1']['recall'])
        mlflow.log_metric('f1_macro_avg',report['macro avg']['f1-score'])
        
        if 'XGB' in model_name:
            mlflow.xgboost.log_model(model,"model")
        else:
            mlflow.sklearn.log_model(model,"model")

2026/03/25 17:31:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 17:31:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/1/runs/cf09dfe15a5e4d66828d15940af38409
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/03/25 17:31:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 17:31:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/1/runs/46f05bf9c2ff4ddcaad31a9949ab8afc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/03/25 17:31:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/669e519dc2c34088b32fbca64f42274c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/03/25 17:31:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier with SMOTE at: http://127.0.0.1:5000/#/experiments/1/runs/a65f4b3563624947a4c5feeb679196de
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


**Start MLflow run**

`with mlflow.start_run(run_name=model_name):`

👉 This creates a new experiment run in MLflow

Each model gets its own run 

run_name=model_name → easy to identify in UI

if 'XGB' in model_name: 

            mlflow.xgboost.log_model(model,"model") 
            
        else: 
        
            mlflow.sklearn.log_model(model,"model")

This checks model type and logs accordingly

XGBoost → mlflow.xgboost 

Others → mlflow.sklearn

👉 Because MLflow uses different modules for different libraries

## Register the Model

In [17]:
model_name="XGB-Smote"
run_id=input("Enter RUN ID:")
model_uri=f"runs:/{run_id}/model"
result = mlflow.register_model(
    model_uri,model_name)

Enter RUN ID:a65f4b3563624947a4c5feeb679196de


Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/03/25 18:33:32 WARNING mlflow.tracking._model_registry.fluent: Run with id a65f4b3563624947a4c5feeb679196de has no artifacts at artifact path 'model', registering model based on models:/m-6e7f2edc95b64d92b79656a7ade2bf2f instead
2026/03/25 18:33:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


## Load the model

In [18]:
model_version=1
model_uri=f"models:/{model_name}/{model_version}"
loaded_model=mlflow.xgboost.load_model(model_uri)
y_pred=loaded_model.predict(x_test)
y_pred[:4]

array([0, 0, 0, 0])

In [19]:
dev_model_uri=f"models:/{model_name}/{model_version}"
prod_model="anomaly-detection-prod"
client=mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri,dst_name=prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1774445317102, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1774445317102, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='a65f4b3563624947a4c5feeb679196de', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [23]:
model_version=1
model_uri=f"models:/{prod_model}/{model_version}"
loaded_model=mlflow.xgboost.load_model(model_uri)
y_pred=loaded_model.predict(x_test)
y_pred[:4]

array([0, 0, 0, 0])